# Tool Calling in LangChain
Tool calling (or function calling) *lets AI models use external tools like APIs, databases, or code to get real-time info or take actions, moving them from passive answer-givers to active agents that can book flights, check weather, or query data by identifying when to use a tool, requesting the call (with parameters), and then processing the tool's structured output to form a final answer.* It's a key AI technique for automation and complex task completion, involving an app defining tools and the LLM deciding when and how to invoke them, with the app handling execution.

- Reference: [CampuX-Colab](https://colab.research.google.com/drive/1-xMYU9ExZqoySEX-XHAvEaE17PCWvc9H?usp=sharing#scrollTo=vBWmBt2cukKY)

In [2]:
!pip install -q --upgrade pip
!pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    safetensors \
    langchain-huggingface \
    langchain

# Import Libraries

In [3]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

In [4]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

In [5]:
from langchain.tools import tool

In [6]:
import torch

In [7]:
from accelerate import init_empty_weights, load_checkpoint_and_dispatch

# Load Model

In [9]:
# Simpler open-source model with no license restrictions
#model_name = "EleutherAI/gpt-neo-125M"  # Smallest, ~500MB
model_name = "unsloth/Qwen2.5-7B-Instruct"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # Use bfloat16 for better performance on modern GPUs
)

hf_llm = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")

print("Model loaded successfully!")

pipe = pipeline(
    "text-generation",
    model=hf_llm,
    tokenizer=tokenizer,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.3,
    top_p=0.95,
    repetition_penalty=1.15,
    return_full_text=False,              # 🔥 REQUIRED
    eos_token_id=tokenizer.eos_token_id,
    device_map="auto"  # keep weights offloaded
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

Device set to use cuda:0


Model loaded successfully!


In [ ]:
llm_model = HuggingFacePipeline(
    pipeline=pipe,
    pipeline_kwargs={
        "stop_sequences": ["<|im_start|>", "<|im_end|>"]  # 🔥 REQUIRED
    }
)

chat_model = ChatHuggingFace(
    llm=llm_model,
    tokenizer=tokenizer,
    chat_template=False                 # 🔥 REQUIRED
)

In [ ]:
# del pipe, tokenizer, hf_llm
# del llm_model, chat_model

In [10]:
chat_model.invoke("What is the capital of France?")

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={}, id='lc_run--019b2b00-59a1-7571-9b7e-5420f19a1e3c-0')

In [11]:
chat_model.invoke([SystemMessage(content=''), HumanMessage(content="What is the capital of France?")])

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={}, id='lc_run--019b2b00-681a-74d1-b608-2a07b6aaac79-0')

# Tool Creation

In [12]:
@tool
def multiply(num1: int, num2: int) -> int:
    """Takes 2 integer input, num1 and num2 then return a integer that Multiplies two numbers num1 and num2"""
    return num1 * num2

@tool
def addition(num1: int, num2: int) -> int:
    """Takes 2 integer input, num1 and num2 then return a integer that Adds two numbers num1 and num2"""
    return num1 + num2

# Tool Binding

In [13]:
tools_model = chat_model.bind_tools([multiply, addition])

In [15]:
query = HumanMessage(content="What is the multiplication of 2 and 3?")

In [16]:
tools_model.invoke([query])

AIMessage(content='The multiplication of 2 and 3 is 6. \n\nIn mathematical terms, this can be written as:\n\n\\[ 2 \\times 3 = 6 \\]', additional_kwargs={}, response_metadata={}, id='lc_run--019b2b00-e376-72a1-a393-306ba4263870-0')

In [ ]:
tools_model.invoke(['what is ai'])

Note: No Tools founds with this LLM model, move to the GEMINI closed source model

# Gemini

In [22]:
# Install the SDK
!pip install -U google-generativeai langchain-google-genai

# Import necessary libraries
import google.generativeai as genai
from google.colab import userdata

# Configure the SDK with the API key from secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

  Attempting uninstall: google-generativeai
    Found existing installation: google-generativeai 0.8.5
    Uninstalling google-generativeai-0.8.5:
      Successfully uninstalled google-generativeai-0.8.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [google-generativeai]


In [27]:
import os
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [29]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
)

In [30]:
llm.invoke("What is the capital of France?")

AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b2b18-bf94-7612-8561-6c48d3805f4b-0', usage_metadata={'input_tokens': 8, 'output_tokens': 29, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 21}})

## Tools Creation

In [31]:
@tool
def multiply(num1: int, num2: int) -> int:
    """Takes 2 integer input, num1 and num2 then return a integer that Multiplies two numbers num1 and num2"""
    return num1 * num2

@tool
def addition(num1: int, num2: int) -> int:
    """Takes 2 integer input, num1 and num2 then return a integer that Adds two numbers num1 and num2"""
    return num1 + num2

# Tools Binding

In [32]:
tools_model = llm.bind_tools([multiply, addition])

In [34]:
tools_model.invoke("What is the capital of France?")

AIMessage(content="I can't answer that question. I can only perform operations using the `multiply` and `addition` functions. ", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b2b20-973f-7652-bde0-509f4538c148-0', usage_metadata={'input_tokens': 148, 'output_tokens': 24, 'total_tokens': 172, 'input_token_details': {'cache_read': 0}})

In [35]:
tools_model.invoke("What is the multiplication of 2 and 3?")

AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"num1": 2, "num2": 3}'}, '__gemini_function_call_thought_signatures__': {'f8f51e38-bd54-4530-83ef-eb62383be0b2': 'CqMCAXLI2nzRLRihT/oFdLXM8uIgWSQIYAWaYTdk5RRooXoQJuFyKsVtAWP8ilFgCedQWIdwMaTQ5+WznsdKBIzsU7uM6wBK1YXc/1hTBbdGYqmLFMJ9spsAj34f6N2f6e+tvwGxDFRVa10jturYOPbLvqWyvDTQWKlEELAoi8UwAR0Ri5NytCfc1TSO12u/1MQAl+4J/hn9/zDLXoSrDAKZVjQ365Z5rG7fTBfla+rgxfsy+J2DoyxnsIWPAuHbOjvLEIG/ZDHqn3/LKQnMAoJSXv4+104Khsik0siDRbk0ahyLluBIXoYz91+CZEDc5i2NHyf5a6s0mf8oO/9icADWMMFXpDJKo6DcMVWAeBKdni9F5wiRvYiF90tZD+XcM+8M6GGJ'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b2b21-5210-7012-9050-651fcabc3697-0', tool_calls=[{'name': 'multiply', 'args': {'num1': 2, 'num2': 3}, 'id': 'f8f51e38-bd54-4530-83ef-eb62383be0b2', 'type': 'tool_call'}], usage_metadata={'input_tokens': 152, 'output_tokens': 89, 'total_tokens':

# Tool Calling

In [36]:
query = HumanMessage(content="What is the multiplication of 2 and 3?")

## Input Message/History build in message

In [37]:
# history maintain
message = [query]

In [38]:
# AI suggest the tools and parameter
result = tools_model.invoke(message)
message.append(result)
result

AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"num1": 2, "num2": 3}'}, '__gemini_function_call_thought_signatures__': {'49511ed1-56a6-4306-a72f-087d0da30871': 'CpcCAXLI2nwcm/8EvbXFywnjvdRG68iNy2+aZwhk059JZ2vvQDyXrAduCQ4VM10c/rXQAlq1p4gSMEsl8TaJNA3zWpdA0lP/vf0xUy49iLWqItN5WG1g321VHhmxcjL2+bAKyv5L04BIKdnw6vALNSYQK6y9OzVUCuWzvtvWQwalm206sOxO7TXfj+yQAc5y2XvBNUj/uducJ98iqgrYmJ4S9zNzng6f7cERB8ModnMZ4HsJPyxONGsFDBCrSJikfJ9dHMGuEFf0eLncvFbuXofkVw+Qd2UzAQFNyQIkhu6T8BdgI9QnoDPFGOy/pVkmLYn6/sIwod3hGICkSW89GxRexiXxRi3/tLAD0vGgb+wH4QaomQjftBIV'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b2b23-3759-7622-a8e8-246e5c52a5d0-0', tool_calls=[{'name': 'multiply', 'args': {'num1': 2, 'num2': 3}, 'id': '49511ed1-56a6-4306-a72f-087d0da30871', 'type': 'tool_call'}], usage_metadata={'input_tokens': 152, 'output_tokens': 88, 'total_tokens': 240, 'input_tok

In [51]:
# This is the suggestions tools and parameters
result.tool_calls[0]

{'name': 'multiply',
 'args': {'num1': 2, 'num2': 3},
 'id': '49511ed1-56a6-4306-a72f-087d0da30871',
 'type': 'tool_call'}

In [54]:
result.tool_calls[0]['name']

'multiply'

## Tool Mapping

In [55]:
# we have multiple tools, maps each other based on given name
if result.tool_calls[0]['name'] == 'multiply':
    tool = multiply
elif result.tool_calls[0]['name'] == 'addition':
    tool = addition
tool

StructuredTool(name='multiply', description='Takes 2 integer input, num1 and num2 then return a integer that Multiplies two numbers num1 and num2', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x7cbd21d05800>)

## tool message

In [56]:
# tool gives a tool message
tool_result = tool.invoke(result.tool_calls[0])

In [57]:
tool

StructuredTool(name='multiply', description='Takes 2 integer input, num1 and num2 then return a integer that Multiplies two numbers num1 and num2', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x7cbd21d05800>)

In [44]:
message.append(tool_result)

In [47]:
display(len(message))
display(message)

3

[HumanMessage(content='What is the multiplication of 2 and 3?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"num1": 2, "num2": 3}'}, '__gemini_function_call_thought_signatures__': {'49511ed1-56a6-4306-a72f-087d0da30871': 'CpcCAXLI2nwcm/8EvbXFywnjvdRG68iNy2+aZwhk059JZ2vvQDyXrAduCQ4VM10c/rXQAlq1p4gSMEsl8TaJNA3zWpdA0lP/vf0xUy49iLWqItN5WG1g321VHhmxcjL2+bAKyv5L04BIKdnw6vALNSYQK6y9OzVUCuWzvtvWQwalm206sOxO7TXfj+yQAc5y2XvBNUj/uducJ98iqgrYmJ4S9zNzng6f7cERB8ModnMZ4HsJPyxONGsFDBCrSJikfJ9dHMGuEFf0eLncvFbuXofkVw+Qd2UzAQFNyQIkhu6T8BdgI9QnoDPFGOy/pVkmLYn6/sIwod3hGICkSW89GxRexiXxRi3/tLAD0vGgb+wH4QaomQjftBIV'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b2b23-3759-7622-a8e8-246e5c52a5d0-0', tool_calls=[{'name': 'multiply', 'args': {'num1': 2, 'num2': 3}, 'id': '49511ed1-56a6-4306-a72f-087d0da30871', 't

# Tool Execution

In [48]:
final_result = tools_model.invoke(message)
final_result

AIMessage(content='The multiplication of 2 and 3 is 6.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b2b29-6495-7651-956e-09bc1ff1d088-0', usage_metadata={'input_tokens': 185, 'output_tokens': 12, 'total_tokens': 197, 'input_token_details': {'cache_read': 0}})

In [49]:
final_result.text

'The multiplication of 2 and 3 is 6.'